# Notebook 4 — Model Deployment with FastAPI
**BSE Stock Market Volatility Forecasting**

**Goals:**
1. Understand the FastAPI server in `src/main.py`
2. Walk through the Pydantic data classes (`FitIn`, `FitOut`, `PredictIn`, `PredictOut`)
3. Start the server and hit `/hello`
4. Call `/fit` to train and save a GARCH model via the API
5. Call `/predict` to get a live volatility forecast
6. Explore the auto-generated interactive docs

> **Note:** This notebook assumes the FastAPI server is running.
> Start it with:
> ```
> cd src
> uvicorn main:app --reload --workers 1 --host localhost --port 8008
> ```
> In Replit the server is started via the workflow panel.

## 1. Setup

In [ ]:
import requests
import json
import pandas as pd

BASE_URL = 'http://localhost:8008'

print(f'Will connect to: {BASE_URL}')
print("Make sure the server is running before executing the cells below.")

## 2. Understanding `main.py`

Here is a summary of what each part of `src/main.py` does:

| Component | Purpose |
|---|---|
| `FastAPI()` | Creates the app instance |
| `FitIn` | Pydantic model for `/fit` request body |
| `FitOut` | Pydantic model for `/fit` response (inherits `FitIn`) |
| `PredictIn` | Pydantic model for `/predict` request body |
| `PredictOut` | Pydantic model for `/predict` response (inherits `PredictIn`) |
| `GET /hello` | Health check |
| `POST /fit` | Fetch data → fit GARCH → save model → return metadata |
| `POST /predict` | Load latest model → forecast → return volatility dict |

Let's look at the key Pydantic models:

In [ ]:
# ── Pydantic data classes (mirrors src/main.py) ──────────────────────────────
from pydantic import BaseModel
from typing import Optional

class FitIn(BaseModel):
    ticker: str
    start_date: str
    end_date: str
    n_observations: int = 2500
    p: int = 1
    q: int = 1

class FitOut(FitIn):
    success: bool
    message: str
    model_path: str
    aic: float
    bic: float

class PredictIn(BaseModel):
    ticker: str
    n_days: int = 5

class PredictOut(PredictIn):
    success: bool
    model_path: str
    forecast: dict

# ── Validate example payloads ──────────────────────────────────────────────
fit_in = FitIn(
    ticker='RELIANCE.NS',
    start_date='2018-01-01',
    end_date='2024-12-31',
    n_observations=1500,
    p=1, q=1
)
print('FitIn model (serialised):')
print(json.dumps(fit_in.model_dump(), indent=2))

predict_in = PredictIn(ticker='RELIANCE.NS', n_days=5)
print('\nPredictIn model:')
print(json.dumps(predict_in.model_dump(), indent=2))

## 3. `/hello` — Health Check

A simple GET request to confirm the server is alive.

In [ ]:
response = requests.get(f'{BASE_URL}/hello')

print(f'Status code : {response.status_code}')
print(f'Response    : {json.dumps(response.json(), indent=2)}')

## 4. `/fit` — Train and Save a GARCH Model

This endpoint:
1. Downloads data from Yahoo Finance for the given `ticker`
2. Stores it in SQLite
3. Fits a GARCH(p, q) model
4. Saves the model to `models/<ticker>_<date>.pkl`
5. Returns the model's AIC, BIC, and file path

In [ ]:
FIT_PAYLOAD = {
    'ticker': 'RELIANCE.NS',
    'start_date': '2018-01-01',
    'end_date': '2024-12-31',
    'n_observations': 1500,
    'p': 1,
    'q': 1
}

print('Sending /fit request (this may take ~15–30 seconds for data download + fitting)...')
response = requests.post(f'{BASE_URL}/fit', json=FIT_PAYLOAD)

print(f'\nStatus code : {response.status_code}')
result = response.json()
print(json.dumps(result, indent=2))

In [ ]:
# Parse the FitOut response into a Pydantic model for type-safe access
if response.status_code == 200:
    fit_out = FitOut(**result)
    print(f'Success     : {fit_out.success}')
    print(f'Ticker      : {fit_out.ticker}')
    print(f'AIC         : {fit_out.aic}')
    print(f'BIC         : {fit_out.bic}')
    print(f'Model path  : {fit_out.model_path}')

## 5. Fit More Stocks

In [ ]:
# Fit models for multiple BSE/NSE tickers
TICKERS_TO_FIT = ['TCS.NS', 'INFY.NS', '^BSESN']

fit_results = []
for ticker in TICKERS_TO_FIT:
    payload = {
        'ticker': ticker,
        'start_date': '2018-01-01',
        'end_date': '2024-12-31',
        'n_observations': 1500,
        'p': 1, 'q': 1
    }
    print(f'Fitting {ticker}...')
    r = requests.post(f'{BASE_URL}/fit', json=payload)
    data = r.json()
    if r.status_code == 200:
        fit_results.append({'ticker': ticker, 'aic': data['aic'], 'bic': data['bic'], 'success': data['success']})
        print(f'  OK — AIC={data["aic"]:.2f}, BIC={data["bic"]:.2f}')
    else:
        print(f'  ERROR: {data}')

print('\nAll models fitted:')
pd.DataFrame(fit_results)

## 6. `/predict` — Forecast Volatility

This endpoint:
1. Finds the most recent saved model for `ticker`
2. Loads it from disk
3. Produces an annualised volatility forecast for `n_days` trading days ahead
4. Returns the forecast as a dict (`h.1`, `h.2`, …, `h.n`)

In [ ]:
PREDICT_PAYLOAD = {
    'ticker': 'RELIANCE.NS',
    'n_days': 5
}

response = requests.post(f'{BASE_URL}/predict', json=PREDICT_PAYLOAD)

print(f'Status code : {response.status_code}')
result = response.json()
print(json.dumps(result, indent=2))

In [ ]:
if response.status_code == 200:
    predict_out = PredictOut(**result)
    print(f'Ticker: {predict_out.ticker}')
    print(f'Model : {predict_out.model_path}')
    print()
    print('Annualised Volatility Forecast:')
    for day_label, vol in predict_out.forecast.items():
        print(f'  {day_label} : {vol:.2f}%')

## 7. Compare Forecasts Across Stocks

In [ ]:
import plotly.graph_objects as go

tickers_to_predict = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', '^BSESN']
forecast_data = {}

for ticker in tickers_to_predict:
    r = requests.post(f'{BASE_URL}/predict', json={'ticker': ticker, 'n_days': 5})
    if r.status_code == 200:
        forecast_data[ticker] = r.json()['forecast']
    else:
        print(f'Prediction failed for {ticker}: {r.json()}')

# Plot
fig = go.Figure()
days = list(range(1, 6))

for ticker, forecast in forecast_data.items():
    vols = list(forecast.values())
    fig.add_trace(go.Scatter(
        x=[f'Day {d}' for d in days],
        y=vols,
        mode='lines+markers',
        name=ticker
    ))

fig.update_layout(
    title='5-Day GARCH Volatility Forecast — BSE/NSE Stocks',
    xaxis_title='Forecast Horizon',
    yaxis_title='Annualised Volatility (%)',
    template='plotly_white',
    height=430
)
fig.show()

## 8. Error Handling

The API returns informative errors when things go wrong.

In [ ]:
# Try predicting for a ticker that hasn't been fitted yet
r = requests.post(f'{BASE_URL}/predict', json={'ticker': 'NOTFITTED.NS', 'n_days': 5})
print(f'Status code : {r.status_code}  (expected 404)')
print(f'Detail      : {r.json()["detail"]}')

In [ ]:
# Try fitting an invalid ticker
r = requests.post(f'{BASE_URL}/fit', json={
    'ticker': 'INVALID_TICKER_XYZ',
    'start_date': '2023-01-01',
    'end_date': '2024-01-01',
})
print(f'Status code : {r.status_code}  (expected 400)')
print(f'Detail      : {r.json()["detail"]}')

## 9. Interactive API Docs

FastAPI auto-generates two documentation UIs:

- **Swagger UI**: http://localhost:8008/docs — try endpoints interactively
- **ReDoc**: http://localhost:8008/redoc — clean reference documentation

Open these in your browser while the server is running.

In [ ]:
# Fetch and display the OpenAPI schema
schema = requests.get(f'{BASE_URL}/openapi.json').json()
print('API title    :', schema['info']['title'])
print('API version  :', schema['info']['version'])
print('Paths        :', list(schema['paths'].keys()))

## Summary

We have built a complete end-to-end volatility forecasting system:

| Step | Tool | Output |
|---|---|---|
| Data collection | `yfinance` | Clean DataFrame with returns |
| Data storage | `SQLite` + `SQLAlchemy` | Persistent stock price DB |
| Volatility modelling | `arch` GARCH(1,1) | Conditional volatility |
| Model persistence | `joblib` | `.pkl` model checkpoints |
| API deployment | `FastAPI` + `uvicorn` | Live prediction endpoint |

**The API is production-ready:**
- Input validation via Pydantic
- Structured error responses
- Auto-generated OpenAPI docs
- Stateless `POST /predict` — just send the ticker, get the forecast

**Run tests:** `pytest tests/ -v`